In [3]:
import sys
print(sys.executable)


c:\Users\Smart\Desktop\mini-gpt-from-scratch\.venv-1\Scripts\python.exe


In [4]:
import torch
print(torch.__version__)

c:\Users\Smart\Desktop\mini-gpt-from-scratch\.venv-1\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


2.14.1+cpu


In [5]:
text = """
I love machine learning.
I love deep learning.
Machine learning is amazing.
Deep learning is powerful.
I study artificial intelligence.
Artificial intelligence is interesting.
"""

In [6]:
tokens = text.split()

In [7]:
print(tokens)

['I', 'love', 'machine', 'learning.', 'I', 'love', 'deep', 'learning.', 'Machine', 'learning', 'is', 'amazing.', 'Deep', 'learning', 'is', 'powerful.', 'I', 'study', 'artificial', 'intelligence.', 'Artificial', 'intelligence', 'is', 'interesting.']


In [8]:
import re

tokens = re.findall(r"\w+|[^\w\s]", text)

print(tokens)

['I', 'love', 'machine', 'learning', '.', 'I', 'love', 'deep', 'learning', '.', 'Machine', 'learning', 'is', 'amazing', '.', 'Deep', 'learning', 'is', 'powerful', '.', 'I', 'study', 'artificial', 'intelligence', '.', 'Artificial', 'intelligence', 'is', 'interesting', '.']


In [9]:
vocab = sorted(set(tokens))

print(vocab)

['.', 'Artificial', 'Deep', 'I', 'Machine', 'amazing', 'artificial', 'deep', 'intelligence', 'interesting', 'is', 'learning', 'love', 'machine', 'powerful', 'study']


In [10]:
print(len(vocab))

16


In [11]:
stoi = {token: i for i, token in enumerate(vocab)}
print(stoi)

{'.': 0, 'Artificial': 1, 'Deep': 2, 'I': 3, 'Machine': 4, 'amazing': 5, 'artificial': 6, 'deep': 7, 'intelligence': 8, 'interesting': 9, 'is': 10, 'learning': 11, 'love': 12, 'machine': 13, 'powerful': 14, 'study': 15}


In [12]:
print(stoi["machine"])

13


In [13]:
itos = {i: token for i, token in enumerate(vocab)}
print(itos)

{0: '.', 1: 'Artificial', 2: 'Deep', 3: 'I', 4: 'Machine', 5: 'amazing', 6: 'artificial', 7: 'deep', 8: 'intelligence', 9: 'interesting', 10: 'is', 11: 'learning', 12: 'love', 13: 'machine', 14: 'powerful', 15: 'study'}


In [14]:
print(itos[7])

deep


In [15]:
def encode(tokens):
    return [stoi[token] for token in tokens]
sample_tokens = ["I", "love", "machine", "learning", "."]
sample_ids = encode(sample_tokens)

print(sample_ids)

[3, 12, 13, 11, 0]


In [16]:
import torch
import torch.nn as nn
embedding = nn.Embedding(len(vocab), 8)
#كل Token هيتحول إلى Vector مكون من 8 أرقام.

In [17]:
machine_id = stoi["machine"]

print(machine_id)

13


In [18]:
machine_id = torch.tensor(machine_id)
machine_vector = embedding(machine_id)

print(machine_vector)

tensor([ 0.4643,  0.4170, -1.4441, -0.8070,  0.5482,  0.3216,  0.5827,  0.6547],
       grad_fn=<EmbeddingBackward0>)


In [19]:
sentence_tokens = ["I", "love", "machine", "learning", "."]
sentence_ids = torch.tensor(encode(sentence_tokens))

print(sentence_ids)

tensor([ 3, 12, 13, 11,  0])


In [20]:
sentence_embeddings = embedding(sentence_ids)

print(sentence_embeddings)

tensor([[ 1.3620, -1.7140,  0.0197, -1.0935, -0.3402,  0.8958,  0.6550,  0.9198],
        [ 0.4388, -0.3461, -0.0668, -0.0445, -1.4162, -0.3262,  0.7759,  2.1100],
        [ 0.4643,  0.4170, -1.4441, -0.8070,  0.5482,  0.3216,  0.5827,  0.6547],
        [-1.2262, -2.9330,  1.3082, -1.7736, -0.2575, -0.6691, -0.5223,  0.4430],
        [ 0.1031,  0.0051,  1.0172, -2.2610,  0.1601,  0.0553,  2.0683, -0.5725]],
       grad_fn=<EmbeddingBackward0>)


In [21]:
sentence_tokens = ["I", "love", "machine", "learning", "."]

sentence_ids = torch.tensor(encode(sentence_tokens))

print(sentence_ids)

sentence_embeddings = embedding(sentence_ids)

print(sentence_embeddings)
print(sentence_embeddings.shape)

tensor([ 3, 12, 13, 11,  0])
tensor([[ 1.3620, -1.7140,  0.0197, -1.0935, -0.3402,  0.8958,  0.6550,  0.9198],
        [ 0.4388, -0.3461, -0.0668, -0.0445, -1.4162, -0.3262,  0.7759,  2.1100],
        [ 0.4643,  0.4170, -1.4441, -0.8070,  0.5482,  0.3216,  0.5827,  0.6547],
        [-1.2262, -2.9330,  1.3082, -1.7736, -0.2575, -0.6691, -0.5223,  0.4430],
        [ 0.1031,  0.0051,  1.0172, -2.2610,  0.1601,  0.0553,  2.0683, -0.5725]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([5, 8])


In [22]:
sentence_tokens = ["I", "love", "machine", "learning", "."]

sentence_ids = torch.tensor(encode(sentence_tokens))

print(sentence_ids)

tensor([ 3, 12, 13, 11,  0])


In [23]:
block_size = len(sentence_ids)

position_embedding = nn.Embedding(block_size, 8)

positions = torch.arange(block_size)
pos_embeddings = position_embedding(positions)

x = sentence_embeddings + pos_embeddings

print("Token embeddings:", sentence_embeddings.shape)
print("Position embeddings:", pos_embeddings.shape)
print("Final x:", x.shape)

Token embeddings: torch.Size([5, 8])
Position embeddings: torch.Size([5, 8])
Final x: torch.Size([5, 8])


In [25]:
import math
d_model = 8

Wq = nn.Linear(d_model, d_model, bias=False)
Wk = nn.Linear(d_model, d_model, bias=False)
Wv = nn.Linear(d_model, d_model, bias=False)

Q = Wq(x)
K = Wk(x)
V = Wv(x)

print("x:", x.shape)
print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

scores = Q @ K.T

print("Scores:", scores.shape)

scores = scores / math.sqrt(d_model)

attention_weights = torch.softmax(scores, dim=-1)

print("Attention weights:", attention_weights.shape)

out = attention_weights @ V

print("Output:", out.shape)

x: torch.Size([5, 8])
Q: torch.Size([5, 8])
K: torch.Size([5, 8])
V: torch.Size([5, 8])
Scores: torch.Size([5, 5])
Attention weights: torch.Size([5, 5])
Output: torch.Size([5, 8])


In [26]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size, dropout):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # True means that this position cannot be attended to.
        mask = torch.triu(
            torch.ones(block_size, block_size, dtype=torch.bool),
            diagonal=1
        )
        self.register_buffer("mask", mask)

        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        T = x.size(1)

        out, _ = self.attention(
            x, x, x,
            attn_mask=self.mask[:T, :T],
            need_weights=False
        )

        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, block_size, dropout):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(
            d_model, num_heads, block_size, dropout
        )

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        num_heads=4,
        num_layers=2,
        block_size=8,
        dropout=0.1
    ):
        super().__init__()

        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    d_model, num_heads, block_size, dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        token_emb = self.token_embedding(idx)
        positions = torch.arange(T, device=idx.device)
        position_emb = self.position_embedding(positions)

        x = token_emb + position_emb
        x = self.blocks(x)
        x = self.final_ln(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=20, temperature=1.0):
        self.eval()

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]

            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / temperature
            probabilities = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)

        return idx

In [28]:
tokens = re.findall(r"\w+|[^\w\s]", text)

vocab = sorted(set(tokens))
stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for i, token in enumerate(vocab)}

def encode(tokens):
    return [stoi[token] for token in tokens]

def decode(ids):
    return [itos[int(i)] for i in ids]

data = torch.tensor(encode(tokens), dtype=torch.long)

print("Data:", data)
print("Data length:", len(data))

Data: tensor([ 3, 12, 13, 11,  0,  3, 12,  7, 11,  0,  4, 11, 10,  5,  0,  2, 11, 10,
        14,  0,  3, 15,  6,  8,  0,  1,  8, 10,  9,  0])
Data length: 30


In [29]:
block_size = 8
batch_size = 16

def get_batch():
    starts = torch.randint(
        0,
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in starts
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in starts
    ])

    return x, y


xb, yb = get_batch()

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("Input tokens:", decode(xb[0].tolist()))
print("Target tokens:", decode(yb[0].tolist()))

Input shape: torch.Size([16, 8])
Target shape: torch.Size([16, 8])
Input tokens: ['learning', 'is', 'amazing', '.', 'Deep', 'learning', 'is', 'powerful']
Target tokens: ['is', 'amazing', '.', 'Deep', 'learning', 'is', 'powerful', '.']


In [31]:
import torch.nn.functional as F

In [32]:
model = MiniGPT(
    vocab_size=len(vocab),
    d_model=32,
    num_heads=4,
    num_layers=2,
    block_size=block_size,
    dropout=0.1
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

Logits shape: torch.Size([16, 8, 16])
Initial loss: 2.918088674545288


In [33]:
num_steps = 1000

model.train()

for step in range(num_steps):
    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (step + 1) % 100 == 0:
        print(
            f"Step {step + 1}/{num_steps} | "
            f"Loss: {loss.item():.4f}"
        )

Step 100/1000 | Loss: 0.4947
Step 200/1000 | Loss: 0.2247
Step 300/1000 | Loss: 0.1266
Step 400/1000 | Loss: 0.1177
Step 500/1000 | Loss: 0.1464
Step 600/1000 | Loss: 0.0863
Step 700/1000 | Loss: 0.1173
Step 800/1000 | Loss: 0.1290
Step 900/1000 | Loss: 0.1128
Step 1000/1000 | Loss: 0.1321


In [34]:
model.eval()

prompt = "I love"
prompt_tokens = re.findall(r"\w+|[^\w\s]", prompt)

prompt_ids = encode(prompt_tokens)

context = torch.tensor(
    [prompt_ids],
    dtype=torch.long
)

generated_ids = model.generate(
    context,
    max_new_tokens=20,
    temperature=0.8
)

generated_tokens = decode(generated_ids[0].tolist())

print("Generated text:", " ".join(generated_tokens))

Generated text: I love deep learning . Machine learning is amazing . Deep learning is powerful . I study artificial intelligence . Artificial intelligence


In [35]:
import json
from pathlib import Path

# Create a folder for the trained model files
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

# Save the trained model and its configuration
checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": len(vocab),
        "d_model": 32,
        "num_heads": 4,
        "num_layers": 2,
        "block_size": block_size,
        "dropout": 0.1,
    },
}

torch.save(checkpoint, artifacts_dir / "mini_gpt.pth")

# Save the vocabulary used during training
with open(artifacts_dir / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(
        {"stoi": stoi, "itos": itos},
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Model and vocabulary saved successfully!")
print("Files:", list(artifacts_dir.iterdir()))

Model and vocabulary saved successfully!
Files: [WindowsPath('artifacts/mini_gpt.pth'), WindowsPath('artifacts/vocab.json')]
